# One-click Colab GPU test — Baseline, V1, V2

Select **Runtime → Change runtime type → T4 GPU**, then choose **Run all**. The default route prepares an isolated CUDA 13 environment, runs the mandatory tests and V1/V2 benchmarks, and creates an evidence ZIP. V3 is intentionally outside this notebook's scope.

The detector section is optional because it needs a local `yolov5s.pt` weight.

In [ ]:
!nvidia-smi


In [ ]:
%%bash
set -euo pipefail
COMMIT=main
REPO=/content/cuda-nms-numba

if [ ! -d "$REPO/.git" ]; then
  git clone https://github.com/liltommy142/cuda-nms-numba.git "$REPO"
fi

cd "$REPO"
git fetch origin
git checkout "$COMMIT"
if [ "$COMMIT" = main ]; then
  git reset --hard origin/main
fi
git rev-parse HEAD


In [ ]:
%%bash
set -euo pipefail
python -m pip install -q virtualenv
python -m virtualenv --clear /content/nms-cu13-venv

/content/nms-cu13-venv/bin/python -m pip install --no-cache-dir \
  "numpy==1.26.4" "pytest==9.1.1" "numba-cuda[cu13]"

/content/nms-cu13-venv/bin/python -m pip install --no-cache-dir \
  "torch==2.5.1" "torchvision==0.20.1" \
  --index-url https://download.pytorch.org/whl/cpu


In [ ]:
%%bash
set -euo pipefail
cd /content/cuda-nms-numba
PYTHONPATH="$PWD/src" /content/nms-cu13-venv/bin/python - <<'PY'
import numpy as np
from numba import cuda
from gpu_v1 import compute_iou_matrix_gpu

print('CUDA module:', cuda.__file__)
assert cuda.is_available(), 'Enable a T4 GPU runtime, then reconnect.'
boxes = np.array([[0, 0, 2, 2], [1, 1, 3, 3]], dtype=np.float32)
iou = compute_iou_matrix_gpu(boxes)
assert np.allclose(np.diag(iou), 1.0)
print('V1 CUDA JIT smoke test: PASS')
PY


In [ ]:
%%bash
set -euo pipefail
cd /content/cuda-nms-numba
COMMIT=$(git rev-parse --short HEAD)
EVIDENCE_DIR="/content/evidence/$COMMIT"
mkdir -p "$EVIDENCE_DIR"

PYTHON=/content/nms-cu13-venv/bin/python
PYTHONPATH="$PWD/src" "$PYTHON" -m pytest tests/test_correctness.py -q -rs -k "cpu or gpu_v1 or gpu_v2" \
  2>&1 | tee "$EVIDENCE_DIR/pytest_cuda.txt"

PYTHONPATH="$PWD/src" "$PYTHON" -m pytest \
  tests/baseline tests/common tests/compat tests/v1 \
  tests/test_benchmarks.py tests/test_colab_notebooks.py -q -rs \
  2>&1 | tee -a "$EVIDENCE_DIR/pytest_cuda.txt"


In [ ]:
%%bash
set -euo pipefail
cd /content/cuda-nms-numba
COMMIT=$(git rev-parse --short HEAD)
EVIDENCE_DIR="/content/evidence/$COMMIT"
mkdir -p "$EVIDENCE_DIR"
PYTHONPATH="$PWD/src"
export PYTHONPATH
PYTHON=/content/nms-cu13-venv/bin/python

"$PYTHON" benchmarks/run_all.py --versions cpu v1 v2 --n 100 1000 10000 \
  --warmup 2 --repeats 7 --seed 0 \
  --json "$EVIDENCE_DIR/benchmark_v1_v2.json" \
  | tee "$EVIDENCE_DIR/benchmark_v1_v2.txt"

"$PYTHON" benchmarks/run_v2_batch.py --batch-size 32 --n 10000 \
  --warmup 2 --repeats 7 --seed 0 \
  --json "$EVIDENCE_DIR/batch32_v2.json" \
  | tee "$EVIDENCE_DIR/batch32_v2.txt"


In [ ]:
%%bash
set -euo pipefail
cd /content/cuda-nms-numba
COMMIT=$(git rev-parse --short HEAD)
EVIDENCE_DIR="/content/evidence/$COMMIT"
PYTHON=/content/nms-cu13-venv/bin/python

{
  git rev-parse HEAD
  nvidia-smi
  "$PYTHON" --version
  "$PYTHON" -m pip show numba numba-cuda torch torchvision
} > "$EVIDENCE_DIR/environment.txt"

cd /content/evidence
rm -f "$COMMIT.zip"
zip -qr "$COMMIT.zip" "$COMMIT"
echo "Evidence: /content/evidence/$COMMIT"
echo "Archive: /content/evidence/$COMMIT.zip"


In [ ]:
%%bash
set -euo pipefail
RUN_DETECTOR=false
if [ "$RUN_DETECTOR" != true ]; then
  echo 'Detector test skipped. Set RUN_DETECTOR=true after uploading yolov5s.pt.'
  exit 0
fi

cd /content/cuda-nms-numba
test -f yolov5s.pt
COMMIT=$(git rev-parse --short HEAD)
EVIDENCE_DIR="/content/evidence/$COMMIT"
PYTHONPATH="$PWD/src" /content/nms-cu13-venv/bin/python \
  benchmarks/run_detector_pipeline.py \
  --image https://ultralytics.com/images/zidane.jpg \
  --runner v2 --conf-threshold 0.01 --max-candidates 11000 \
  --warmup 1 --repeats 3 \
  --json "$EVIDENCE_DIR/detector_v2_cap11000.json"


In [ ]:
DOWNLOAD_EVIDENCE = False

if DOWNLOAD_EVIDENCE:
    from pathlib import Path
    from google.colab import files

    archives = sorted(Path('/content/evidence').glob('*.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'Expected one evidence ZIP, found: {archives}')
    files.download(str(archives[0]))
else:
    print('Evidence download skipped. Set DOWNLOAD_EVIDENCE = True to download the ZIP.')
